In [61]:
import os
import time
from pinecone import Pinecone, ServerlessSpec
from pinecone_text.sparse import BM25Encoder
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_community.retrievers import PineconeHybridSearchRetriever
from pydantic import SecretStr
from langchain_openrouter import ChatOpenRouter
from dotenv import load_dotenv, find_dotenv

# .env wird nicht automatisch geladen; find_dotenv sucht aufwärts im Verzeichnisbaum
load_dotenv(find_dotenv(usecwd=True), override=True)
# -------------------------------------------------------

True

In [62]:
# 1. API-Key & Pinecone Client initialisieren
# --------------------------------------------------------
api_key = os.environ.get("PINECONE_API_KEY", "IHR_PINECONE_API_KEY")

# api_key = SecretStr(os.getenv("PINECONE_API_KEY", ""))
# if not api_key.get_secret_value():
#     raise RuntimeError("PINECONE_API_KEY is not set")

pc = Pinecone(api_key=api_key)
index_name = "langchain-rag-demo"
# Embedding-Modell laden (Dimension = 384)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [63]:
# Serverless Index anlegen, falls noch nicht vorhanden
# WICHTIG: Fuer Hybrid Search (Dense + Sparse) ist metric="dotproduct" zwingend!
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="dotproduct",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    while not pc.describe_index(index_name).status["ready"]:
        time.sleep(1)
index = pc.Index(index_name)

In [64]:
# --------------------------------------------------------
# 2. Beispieldokumente definieren
# --------------------------------------------------------
docs = [
    Document(
        page_content="Der Fehlercode ERR-4029 tritt auf, wenn das Druckventil überhitzt.",
        metadata={"source": "handbuch_v1.pdf", "kategorie": "Wartung"}
    ),
    Document(
        page_content="Bei einem Temperaturanstieg im Hauptkessel muss das System sofort gekühlt werden.",
        metadata={"source": "sicherheit.pdf", "kategorie": "Sicherheit"}
    ),
    Document(
        page_content="Das Ersatzteil mit der Artikelnummer ART-88912 ist ein Hochdruck Dichtungsring.",
        metadata={"source": "katalog.pdf", "kategorie": "Ersatzteile"}
    ),
    Document(
        page_content="Regelmäßige Inspektionen verhindern Ausfälle der Hydraulikpumpe.",
        metadata={"source": "wartungsplan.pdf", "kategorie": "Wartung"}
    ),
]

In [65]:
# --------------------------------------------------------
# 3. VARIANTE A: Rein Semantischer Retriever (Dense Search)
# --------------------------------------------------------
vectorstore = PineconeVectorStore.from_documents(
    documents=docs,
    embedding=embeddings,
    index_name=index_name,
    namespace="semantisch"
)
semantic_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)


In [66]:
# --------------------------------------------------------
# 4. VARIANTE B: Nativer Hybrider Retriever (Dense + BM25)
# --------------------------------------------------------
# BM25-Encoder auf den Dokumenttexten trainieren und speichern
bm25_encoder = BM25Encoder()
texts = [doc.page_content for doc in docs]
metadatas = [doc.metadata for doc in docs]
bm25_encoder.fit(texts)
bm25_encoder.dump("pinecone_bm25_values.json")
# Hybriden Retriever mit alpha=0.5 (50% Semantik, 50% Keyword) konfigurieren
hybrid_retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25_encoder,
    index=index,
    namespace="hybrid",
    top_k=2,
    alpha=0.5
)

  0%|          | 0/4 [00:00<?, ?it/s]

In [67]:
# Dokumente als kombinierte Dense- und Sparse-Vektoren hochladen
# namespace muss hier explizit gesetzt werden; add_texts nutzt self.namespace nicht
hybrid_retriever.add_texts(texts=texts, metadatas=metadatas, namespace="hybrid")
# Kurze Wartezeit fuer Index-Synchronisation in Pinecone
time.sleep(2)
# --------------------------------------------------------
# 5. Vergleich der Suchergebnisse im Test
# --------------------------------------------------------
query = "Was bedeutet ERR-4029?"
print("=== 1. Semantische Suche (Pinecone) ===")
for doc in semantic_retriever.invoke(query):
    print(f"- {doc.page_content} (Metadaten: {doc.metadata})")
print("\n=== 2. Hybride Suche (Pinecone Native) ===")
for doc in hybrid_retriever.invoke(query):
    print(f"- {doc.page_content} (Metadaten: {doc.metadata})")

  0%|          | 0/1 [00:00<?, ?it/s]

=== 1. Semantische Suche (Pinecone) ===
- Der Fehlercode ERR-4029 tritt auf, wenn das Druckventil überhitzt. (Metadaten: {'kategorie': 'Wartung', 'source': 'handbuch_v1.pdf'})
- Der Fehlercode ERR-4029 tritt auf, wenn das Druckventil überhitzt. (Metadaten: {'kategorie': 'Wartung', 'source': 'handbuch_v1.pdf'})

=== 2. Hybride Suche (Pinecone Native) ===
- Der Fehlercode ERR-4029 tritt auf, wenn das Druckventil überhitzt. (Metadaten: {'kategorie': 'Wartung', 'source': 'handbuch_v1.pdf', 'text': 'Der Fehlercode ERR-4029 tritt auf, wenn das Druckventil überhitzt.', 'score': 4.88532686})
- Das Ersatzteil mit der Artikelnummer ART-88912 ist ein Hochdruck Dichtungsring. (Metadaten: {'kategorie': 'Ersatzteile', 'source': 'katalog.pdf', 'text': 'Das Ersatzteil mit der Artikelnummer ART-88912 ist ein Hochdruck Dichtungsring.', 'score': 2.52533722})


In [68]:
api_key = SecretStr(os.getenv("OPENROUTER_API_KEY", ""))
if not api_key.get_secret_value():
    raise RuntimeError("OPENROUTER_API_KEY is not set")

In [69]:
qwen = "qwen/qwen3-30b-a3b-instruct-2507"


In [70]:
# --------------------------------------------------------
# 6. Einbindung in eine komplette RAG-Chain (LCEL)
# --------------------------------------------------------
def format_docs(retrieved_docs):
    return "\n\n".join(doc.page_content for doc in retrieved_docs)
prompt = ChatPromptTemplate.from_template("""
Beantworte die Frage ausschließlich auf Basis des folgenden Kontexts:
Kontext:
{context}
Frage: {question}
""")
# Beliebiges LangChain-ChatModel einsetzen (z. B. ChatOpenAI oder ChatOllama)
# from langchain_ollama import ChatOllama
llm = ChatOpenRouter(
    api_key=api_key,
    model=qwen,
    temperature=0,
)

In [71]:
def build_rag_chain(retriever, llm):
    return (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

In [72]:
# Beispielaufruf (sobald ein LLM-Objekt uebergeben wird):
rag_chain = build_rag_chain(hybrid_retriever, llm)
antwort = rag_chain.invoke("Was bedeutet ERR-4029?")
print(antwort)

ERR-4029 bedeutet, dass das Druckventil überhitzt ist.
